In [1]:
import torch
from reconstruction import AE
from datasets import MeshData
from utils import utils_m, DataLoader, mesh_sampling, sap
import numpy as np
import pyvista as pv
from skimage import measure
from ipywidgets import interact, interactive, fixed, interact_manual, FloatSlider
from IPython.display import display
import meshplot as mp
import os, sys
from math import ceil
from scipy.ndimage import zoom
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
import os
print(os.getcwd())

/home/jakaria/Explaining_Shape_Variability/src/DeepLearning/compute_canada/guided_vae/reconstruction


In [ ]:
# Meshplot left an annoying print statement in their code. Using this context manager to supress it...
class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

In [ ]:
device = torch.device('cuda', 1)
# Set the path to the saved model directory
#model_path = "/home/jakaria/torus_bump_500_three_scale_binary_bump_variable_noise_fixed_angle/models_classification_regression_only_correlation_loss/models/65"
model_path = "/home/jakaria/Explaining_Shape_Variability/src/DeepLearning/compute_canada/guided_vae/data/CoMA/raw/torus/models_attr_vae/63"
# Load the saved model
model_state_dict = torch.load(f"{model_path}/model_state_dict.pt")
in_channels = torch.load(f"{model_path}/in_channels.pt")
out_channels = torch.load(f"{model_path}/out_channels.pt")
latent_channels = torch.load(f"{model_path}/latent_channels.pt")
spiral_indices_list = torch.load(f"{model_path}/spiral_indices_list.pt")
up_transform_list = torch.load(f"{model_path}/up_transform_list.pt")
down_transform_list = torch.load(f"{model_path}/down_transform_list.pt")
std = torch.load(f"{model_path}/std.pt")
mean = torch.load(f"{model_path}/mean.pt")
template_face = torch.load(f"{model_path}/faces.pt")

# Create an instance of the model
model = AE(in_channels, out_channels, latent_channels,
           spiral_indices_list, down_transform_list,
           up_transform_list)
model.load_state_dict(model_state_dict)
model.to(device)
# Set the model to evaluation mode
model.eval()

In [ ]:
z = torch.zeros(16)
plot=None
@mp.interact(**{f'z[{i}]': FloatSlider(min=-1, max=1, step=0.2, value=0) for i in range(16)})
def show(**kwargs):
    global plot
    global z
    z = torch.tensor([kwargs[f'z[{i}]'] for i in range(16)])
    with torch.no_grad():
        z = z.to(device)
        #print(z)
        pred = model.decoder(z)

        reshaped_pred = (pred.view(-1, 3).cpu() * std) + mean
        reshaped_pred = reshaped_pred.cpu().numpy()
        print(reshaped_pred.shape)

    verts = reshaped_pred
    pcd = o3d.io.read_triangle_mesh('/home/jakaria/Explaining_Shape_Variability/src/DeepLearning/compute_canada/guided_vae/data/CoMA/raw/torus/template/template.ply')
    faces = np.asarray(pcd.triangles)

    if plot is None:
        plot = mp.plot(verts, faces, return_plot=True)
    else:
        with HiddenPrints():
            plot.update_object(vertices=verts, faces=faces)
        display(plot._renderer)

In [ ]:
latent_channels = torch.load(f"{model_path}/latent_channels.pt")
angles = torch.load(f"{model_path}/angles.pt")

In [ ]:
import torch

# Sample flattened labels
y_expanded = torch.tensor([[1.0, 0.95, 0.9, 0.2]])
threshold = 0.05001

abs_diff_matrix = torch.abs(y_expanded - y_expanded.t())
same_class_mask = abs_diff_matrix <= threshold

print(abs_diff_matrix)
print(same_class_mask)


In [ ]:
model_path_root = "/home/jakaria/Explaining_Shape_Variability/src/DeepLearning/compute_canada/guided_vae/data/CoMA/raw/torus/labels.pt"
labels = torch.load(model_path_root)

In [ ]:
labels

In [ ]:
model_path_root = "/home/jakaria/Explaining_Shape_Variability/src/DeepLearning/compute_canada/guided_vae/data/CoMA/raw/hippocampus/labels.pt"
labels = torch.load(model_path_root)

In [ ]:
labels['ab300_001'][1]

In [2]:
model_path_root = "/home/jakaria/Explaining_Shape_Variability/src/DeepLearning/compute_canada/guided_vae/data/CoMA/raw/torus/models_attribute_new"
trials = torch.load(f"{model_path_root}/intermediate_trials.pt")

In [11]:
trials[0]

FrozenTrial(number=0, state=TrialState.COMPLETE, values=[162.88453674316406, 0.32399999999999995, 0.9396163979447569], datetime_start=datetime.datetime(2024, 9, 8, 23, 42, 16, 716685), datetime_complete=datetime.datetime(2024, 9, 8, 23, 48, 21, 924963), params={'sequence_length': 18}, user_attrs={}, system_attrs={'nsga2:generation': 0}, intermediate_values={}, distributions={'sequence_length': IntDistribution(high=50, log=False, low=5, step=1)}, trial_id=0, value=None)

In [3]:
for trial in trials:
    params = trial.params
    values = trial.values
    print(f"Trial ID: {trial.number}")
    print("Parameters:", params)
    print("Values:", values)
    print("-" * 40)

Trial ID: 0
Parameters: {'sequence_length': 18}
Values: [162.88453674316406, 0.32399999999999995, 0.9396163979447569]
----------------------------------------
Trial ID: 1
Parameters: {'sequence_length': 38}
Values: [159.17034912109375, 0.32799999999999996, 0.9562762124023502]
----------------------------------------
Trial ID: 2
Parameters: {'sequence_length': 28}
Values: [166.01568603515625, 0.31399999999999995, 0.9594505365272921]
----------------------------------------
Trial ID: 3
Parameters: {'sequence_length': 41}
Values: [165.33700561523438, 0.28600000000000003, 0.9564343499913834]
----------------------------------------
Trial ID: 4
Parameters: {'sequence_length': 9}
Values: [159.41949462890625, 0.31799999999999995, 0.9542043394273759]
----------------------------------------
Trial ID: 5
Parameters: {'sequence_length': 14}
Values: [158.95547485351562, 0.33799999999999997, 0.9652592651087071]
----------------------------------------
Trial ID: 6
Parameters: {'sequence_length': 17}